# Notebook 01 — 正样本数据清洗

**输入**：`data/raw/2023-2025实施ST.xlsx`（iFind 下载，266 行原始数据）  
**输出**：`data/processed/01_positive_samples.csv`（约 138 家干净正样本）

## 清洗流程
| Step | 内容 | 预期行数 |
|------|------|---------|
| 1 | 读取原始 Excel | ~267 行 |
| 2 | 清理垃圾行 | 266 行 |
| 3 | 解析首次 ST 日期 | 266 行 |
| 4 | 筛选真正首次戴帽 | ~147 行 |
| 5 | 多重剔除 | ~138 行 |
| 6 | 整理输出列 | ~138 行 |
| 7 | 质量检查报告 | — |

In [1]:
# 安装依赖（第一次运行时执行，之后可跳过）
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install",
                "pandas", "openpyxl", "python-calamine", "-q"])


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip


CompletedProcess(args=['/usr/local/bin/python3', '-m', 'pip', 'install', 'pandas', 'openpyxl', 'python-calamine', '-q'], returncode=0)

In [2]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

# ── 路径配置 ──────────────────────────────────────────────────────
PROJECT_ROOT   = Path("..").resolve()
RAW_DATA_DIR   = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR  = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RAW_FILE = RAW_DATA_DIR / "2023-2025实施ST.xlsx"

print(f"原始文件路径: {RAW_FILE}")
print(f"文件存在: {RAW_FILE.exists()}")

原始文件路径: /Users/aa00551/Desktop/Projects/A-Share-ST-Risk-Predictor/data/raw/2023-2025实施ST.xlsx
文件存在: True


## Step 1 — 读取原始 Excel

iFind 导出的 Excel 有两个已知问题：
1. **双层表头**：第 1 行是分组标题（空/实施ST/行业），第 2 行才是真正的列名，读取时要跳过第 1 行
2. **样式兼容问题**：openpyxl 遇到 iFind 特有的颜色格式会报错，改用 `calamine` 引擎绕过

In [3]:
def read_ifind_excel(file_path: Path) -> pd.DataFrame:
    """
    读取 iFind 导出的 Excel，自动处理样式兼容问题。
    优先用 calamine 引擎；失败则回退到 openpyxl（忽略样式警告）。
    """
    # 第 1 行是分组表头，skiprows=1 跳过它，让第 2 行成为列名
    try:
        df = pd.read_excel(file_path, skiprows=1, engine="calamine")
        print("✓ 使用 calamine 引擎读取成功")
    except Exception as e1:
        print(f"calamine 失败: {e1}")
        print("  → 改用 openpyxl 引擎重试...")
        import warnings
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")   # 忽略颜色格式警告
            df = pd.read_excel(file_path, skiprows=1, engine="openpyxl")
        print("✓ openpyxl 引擎读取成功")
    return df


df_raw = read_ifind_excel(RAW_FILE)

print(f"\n总行数: {len(df_raw)}")
print(f"列名清单: {df_raw.columns.tolist()}")
print("\n前 5 行预览:")
df_raw.head()

✓ 使用 calamine 引擎读取成功

总行数: 268
列名清单: ['代码', '名称', '日期', '前简称', '后简称', '原因', '证监会(新)', '同花顺(新)', '申万行业', '首发上市日期', '戴帽摘帽时间']

前 5 行预览:


,代码,名称,日期,前简称,后简称,原因,证监会(新),同花顺(新),申万行业,首发上市日期,戴帽摘帽时间
0,002306.SZ,*ST云网,2025-12-31,*ST云网,*ST云网,主要银行帐号被冻结,互联网和相关服务,餐饮,餐饮,2009-11-11,*ST:20150430;摘帽:20160516;*ST:20170427;摘*:20190...
1,300173.SZ,ST福能,2025-12-23,福能东方,ST福能,根据中国证监会行政处罚事先告知书载明的事实，公司披露的年度报告财务指标存在虚假记载,专用设备制造业,锂电专用设备,锂电专用设备,2011-02-01,ST:20251223
2,002424.SZ,ST百灵,2025-12-23,贵州百灵,ST百灵,根据中国证监会行政处罚事先告知书载明的事实，公司披露的年度报告财务指标存在虚假记载,医药制造业,中药Ⅲ,中药Ⅲ,2010-06-03,ST:20240506;摘帽:20250630;ST:20251223
3,600079.SH,ST人福,2025-12-16,人福医药,ST人福,根据中国证监会行政处罚事先告知书载明的事实，公司披露的年度报告财务指标存在虚假记载,医药制造业,化学制剂,化学制剂,1997-06-06,ST:20251216
4,603822.SH,ST嘉澳,2025-12-11,嘉澳环保,ST嘉澳,根据中国证监会行政处罚事先告知书载明的事实，公司披露的年度报告财务指标存在虚假记载,石油加工、炼焦和核燃料加工业,其他化学制品,其他化学制品,2016-04-28,ST:20251211


## Step 2 — 清理垃圾行

iFind 导出时会在最后一行附上"数据来源：同花顺iFinD"，这不是真实数据，需要删掉。  
判断方法：真实股票的"代码"列格式为 `xxxxxx.SH / .SZ / .BJ`，包含数字；垃圾行不含数字。

In [4]:
# 找到"代码"列（列名可能因 iFind 版本有差异，先确认）
code_col = "代码"

# 只保留"代码"列包含数字的行（真实股票代码）
df = df_raw[df_raw[code_col].astype(str).str.contains(r"\d", na=False)].copy()
df = df.reset_index(drop=True)

print(f"清理前: {len(df_raw)} 行")
print(f"清理后: {len(df)} 行（删除了 {len(df_raw) - len(df)} 行垃圾数据）")
print(f"\n最后 3 行（确认末尾已干净）:")
df.tail(3)

清理前: 268 行
清理后: 266 行（删除了 2 行垃圾数据）

最后 3 行（确认末尾已干净）:


,代码,名称,日期,前简称,后简称,原因,证监会(新),同花顺(新),申万行业,首发上市日期,戴帽摘帽时间
263,300268.SZ,*ST佳沃,2023-04-10,佳沃食品,*ST佳沃,最近一个会计年度经审计的期末净资产为负值或者被追溯重述后为负值,农副食品加工业,其他农产品加工,其他农产品加工,2011-09-27,*ST:20230410;摘*:20240515;*ST:20250314
264,000697.SZ,炼石航空,2023-03-23,炼石航空,*ST炼石,最近一个会计年度经审计的期末净资产为负值或者被追溯重述后为负值,铁路、船舶、航空航天和其他运输设备制造业,航空装备,航空装备Ⅲ,1997-03-25,*ST:20090410;摘帽:20120725;*ST:20230323;摘帽:20240...
265,300555.SZ,ST路通,2023-02-01,路通视信,ST路通,公司向控股股东或其关联方提供资金或违反规定程序对外提供担保且情形严重的,计算机、通信和其他电子设备制造业,通信终端及配件,通信终端及配件,2016-10-18,ST:20230201


## Step 3 — 解析"戴帽摘帽时间"字段，提取首次 ST 日期

`戴帽摘帽时间` 字段记录了完整的 ST 历史，格式如：  
`*ST:20150430;摘帽:20160516;*ST:20170427;摘帽:20210414;*ST:20250416`

用正则 `\d{8}` 提取所有 8 位日期，**取最小值**作为该公司的"历史上第一次被 ST 的日期"。  
这个字段是判断"是否真正首次戴帽"的关键依据。

In [5]:
history_col = "戴帽摘帽时间"

def extract_first_st_date(history_str) -> pd.Timestamp:
    """
    从戴帽摘帽时间字符串中提取历史上第一次 ST 的日期。
    取所有 8 位数字日期中的最小值（最早日期）。
    """
    if pd.isna(history_str):
        return pd.NaT
    # 找出所有 8 位纯数字（YYYYMMDD 格式）
    all_dates = re.findall(r"\d{8}", str(history_str))
    if not all_dates:
        return pd.NaT
    # 转成日期并取最小值（最早的 ST 事件）
    parsed = pd.to_datetime(all_dates, format="%Y%m%d", errors="coerce")
    return parsed.min()


df["首次ST日期"] = df[history_col].apply(extract_first_st_date)

# 统计结果
success_count = df["首次ST日期"].notna().sum()
print(f"成功提取首次 ST 日期: {success_count} 行（共 {len(df)} 行）")
print(f"未能提取: {len(df) - success_count} 行\n")

print("首次 ST 年份分布（含历史）：")
print(df["首次ST日期"].dt.year.value_counts().sort_index())
print("\n⚠️ 注意：很多公司历史上早就被 ST 过，这次只是重新戴帽，Step 4 会过滤掉这些")

成功提取首次 ST 日期: 266 行（共 266 行）
未能提取: 0 行

首次 ST 年份分布（含历史）：
首次ST日期
1998     4
1999     3
2000     3
2001     3
2002     6
2003     2
2004     4
2005     1
2006     3
2007     2
2008     1
2009     3
2011     3
2012     2
2013     2
2014     2
2015     4
2016     4
2017     2
2018     4
2019     5
2020     8
2021     7
2022    10
2023    37
2024    58
2025    83
Name: count, dtype: int64

⚠️ 注意：很多公司历史上早就被 ST 过，这次只是重新戴帽，Step 4 会过滤掉这些


## Step 4 — 筛选"真正首次戴帽"的公司

**判断逻辑：**  
- `日期`（本次实施 ST）== `首次ST日期`（历史最早 ST）→ 这次就是第一次，**保留**  
- `日期` > `首次ST日期` → 历史上早就 ST 过了，这次是重新戴帽，**剔除**

用 `<=` 比较（而非 `==`）是为了容错：iFind 字段里的日期有时与"日期"列存在 1-2 天偏差。

In [6]:
# 把"日期"列统一转为 datetime 类型，方便比较
df["日期"] = pd.to_datetime(df["日期"], errors="coerce")

# 判断：本次实施日期 <= 历史最早 ST 日期 → 真正首次戴帽
df["是否本次为首次戴帽"] = df["日期"] <= df["首次ST日期"]

# 筛选
df_first = df[df["是否本次为首次戴帽"]].copy().reset_index(drop=True)

print(f"筛选前: {len(df)} 行")
print(f"筛选后: {len(df_first)} 行（真正首次戴帽）")
print(f"剔除了: {len(df) - len(df_first)} 家历史上重复 ST 的公司\n")

print("真正首次戴帽按年份分布：")
print(df_first["日期"].dt.year.value_counts().sort_index())

筛选前: 266 行
筛选后: 147 行（真正首次戴帽）
剔除了: 119 家历史上重复 ST 的公司

真正首次戴帽按年份分布：
日期
2023    26
2024    45
2025    76
Name: count, dtype: int64


## Step 5 — 多重剔除

按顺序剔除以下四类公司，每步打印行数变化：
1. **同公司去重**：同一家公司若有多条记录（如 ST 升 \*ST），保留最早一条
2. **北交所**：代码后缀 `.BJ`，流动性差、规则不同
3. **金融业**：证监会行业含"金融/银行/保险/证券/货币"，财务结构与非金融公司完全不同，建模时会干扰
4. **上市不满 3 年**：上市太短的公司历史数据少，且可能是特殊情况

In [7]:
df_clean = df_first.copy()

# ── 5.1 同公司去重：保留最早的实施日期那条 ───────────────────────
before = len(df_clean)
df_clean = (df_clean
            .sort_values("日期")
            .drop_duplicates(subset=["代码"], keep="first")
            .reset_index(drop=True))
print(f"5.1 去重后: {len(df_clean)} 行（剔除 {before - len(df_clean)} 条重复记录）")

# ── 5.2 剔除北交所（代码后缀 .BJ）────────────────────────────────
before = len(df_clean)
df_clean = df_clean[~df_clean["代码"].str.endswith(".BJ")].reset_index(drop=True)
print(f"5.2 剔除北交所后: {len(df_clean)} 行（剔除 {before - len(df_clean)} 家）")

# ── 5.3 剔除金融业 ────────────────────────────────────────────────
# 证监会行业列名以实际列名为准（Step 1 打印的列名清单里确认）
csrc_col = "证监会(新)"
finance_keywords = ["金融", "银行", "保险", "证券", "货币"]
finance_pattern = "|".join(finance_keywords)   # 用 | 连接，表示"或"

before = len(df_clean)
is_finance = df_clean[csrc_col].astype(str).str.contains(finance_pattern, na=False)
print(f"  被识别为金融业的公司: {df_clean[is_finance]['名称'].tolist()}")
df_clean = df_clean[~is_finance].reset_index(drop=True)
print(f"5.3 剔除金融业后: {len(df_clean)} 行（剔除 {before - len(df_clean)} 家）")

# ── 5.4 剔除上市不满 3 年 ─────────────────────────────────────────
ipo_col = "首发上市日期"
df_clean[ipo_col] = pd.to_datetime(df_clean[ipo_col], errors="coerce")

# 计算上市年数（用 ST 日期 - IPO 日期）
df_clean["上市至首次ST年数"] = (
    (df_clean["日期"] - df_clean[ipo_col]).dt.days / 365.25
)

before = len(df_clean)
df_clean = df_clean[df_clean["上市至首次ST年数"] >= 3].reset_index(drop=True)
print(f"5.4 剔除上市不满3年后: {len(df_clean)} 行（剔除 {before - len(df_clean)} 家）")

print(f"\n✓ 最终保留: {len(df_clean)} 家公司")

5.1 去重后: 147 行（剔除 0 条重复记录）
5.2 剔除北交所后: 146 行（剔除 1 家）
  被识别为金融业的公司: ['仁东控股']
5.3 剔除金融业后: 145 行（剔除 1 家）
5.4 剔除上市不满3年后: 138 行（剔除 7 家）

✓ 最终保留: 138 家公司


## Step 6 — 整理输出列，保存 CSV

添加建模需要的派生字段，重命名为英文列名（后续代码更方便），保存到 `data/processed/`。

In [8]:
df_output = df_clean.copy()

# ── 派生字段 ──────────────────────────────────────────────────────
# T 年：ST 发生的年份
df_output["T年"] = df_output["日期"].dt.year

# T-2 年：后续要去下载这几年的财务数据
df_output["T-2年"] = df_output["T年"] - 2

# ST 类型：看"后简称"是否以 *ST 开头
df_output["ST类型"] = df_output["后简称"].apply(
    lambda name: "*ST" if str(name).startswith("*ST") else "ST"
)

# 标签：全部填 1（正样本）
df_output["标签"] = 1

# ── 列重命名（中文 → 英文）────────────────────────────────────────
rename_map = {
    "代码":         "stock_code",
    "名称":         "stock_name",
    "日期":         "first_st_date",
    "T年":          "t_year",
    "T-2年":        "t_minus_2_year",
    "ST类型":       "st_type",
    "原因":         "st_reason",
    "证监会(新)":   "csrc_industry",
    "首发上市日期": "ipo_date",
    "上市至首次ST年数": "years_listed_to_st",
    "戴帽摘帽时间": "st_history",
    "标签":         "label",
}
df_output = df_output.rename(columns=rename_map)

# ── 只保留需要的列 ────────────────────────────────────────────────
keep_cols = list(rename_map.values())
df_output = df_output[keep_cols]

# ── 日期列格式化为字符串 YYYY-MM-DD ──────────────────────────────
df_output["first_st_date"] = df_output["first_st_date"].dt.strftime("%Y-%m-%d")
df_output["ipo_date"]      = df_output["ipo_date"].dt.strftime("%Y-%m-%d")

# ── 按 first_st_date 降序排列 ─────────────────────────────────────
df_output = df_output.sort_values("first_st_date", ascending=False).reset_index(drop=True)

# ── 保存 ──────────────────────────────────────────────────────────
output_path = PROCESSED_DIR / "01_positive_samples.csv"
df_output.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"✓ 已保存: {output_path}")
print(f"  行数: {len(df_output)}, 列数: {len(df_output.columns)}")
print(f"\n前 5 行预览:")
df_output.head()

✓ 已保存: /Users/aa00551/Desktop/Projects/A-Share-ST-Risk-Predictor/data/processed/01_positive_samples.csv
  行数: 138, 列数: 12

前 5 行预览:


,stock_code,stock_name,first_st_date,t_year,t_minus_2_year,st_type,st_reason,csrc_industry,ipo_date,years_listed_to_st,st_history,label
0,300173.SZ,ST福能,2025-12-23,2025,2023,ST,根据中国证监会行政处罚事先告知书载明的事实，公司披露的年度报告财务指标存在虚假记载,专用设备制造业,2011-02-01,14.891170,ST:20251223,1
1,600079.SH,ST人福,2025-12-16,2025,2023,ST,根据中国证监会行政处罚事先告知书载明的事实，公司披露的年度报告财务指标存在虚假记载,医药制造业,1997-06-06,28.528405,ST:20251216,1
2,603822.SH,ST嘉澳,2025-12-11,2025,2023,ST,根据中国证监会行政处罚事先告知书载明的事实，公司披露的年度报告财务指标存在虚假记载,石油加工、炼焦和核燃料加工业,2016-04-28,9.620808,ST:20251211,1
3,300460.SZ,ST惠伦,2025-12-11,2025,2023,ST,根据中国证监会行政处罚事先告知书载明的事实，公司披露的年度报告财务指标存在虚假记载,计算机、通信和其他电子设备制造业,2015-05-15,10.576318,ST:20251211,1
4,002689.SZ,ST远智,2025-12-02,2025,2023,ST,根据中国证监会行政处罚事先告知书载明的事实，公司披露的年度报告财务指标存在虚假记载,通用设备制造业,2012-07-17,13.377139,ST:20251202,1


## Step 7 — 质量检查报告

In [9]:
year_dist    = df_output["t_year"].value_counts().sort_index()
st_type_dist = df_output["st_type"].value_counts()
t2_dist      = df_output["t_minus_2_year"].value_counts().sort_index()
industry_top5 = df_output["csrc_industry"].value_counts().head(5)

print("=" * 56)
print("              正样本质量报告")
print("=" * 56)
print(f"总样本数:        {len(df_output)} 家")
print("\n按年份分布（T 年）:")
for year, cnt in year_dist.items():
    print(f"  {year} 年: {cnt} 家")
print("\n按 ST 类型:")
for st, cnt in st_type_dist.items():
    print(f"  {st}: {cnt} 家")
print("\n按 T-2 年（后续需下载哪几年财务数据）:")
for year, cnt in t2_dist.items():
    print(f"  {year} 年: {cnt} 家")
print("\n行业 Top 5 (证监会):")
for industry, cnt in industry_top5.items():
    print(f"  {industry}: {cnt} 家")
print("=" * 56)

              正样本质量报告
总样本数:        138 家

按年份分布（T 年）:
  2023 年: 24 家
  2024 年: 43 家
  2025 年: 71 家

按 ST 类型:
  ST: 83 家
  *ST: 55 家

按 T-2 年（后续需下载哪几年财务数据）:
  2021 年: 24 家
  2022 年: 43 家
  2023 年: 71 家

行业 Top 5 (证监会):
  计算机、通信和其他电子设备制造业: 15 家
  软件和信息技术服务业: 14 家
  医药制造业: 12 家
  专用设备制造业: 11 家
  建筑装饰和其他建筑业: 7 家
